In [1]:
import pandas as pd
import bioframe as bf

In [2]:
chrom_sizes_file = "/project/fudenber_735/genomes/mm10/mm10.chrom.sizes.reduced"
dot_file = "/project/fudenber_735/GEO/bonev_2017_GSE96107/distiller-0.3.1_mm10/results/coolers/features/mustache_HiC_ES.mm10.mapq_30.10000.tsv"
autosomes_only = True
seq_length = 1310720

In [3]:
if autosomes_only:
    chromID_to_drop = ["chrX", "chrY", "chrM"]

In [4]:
def filter_by_chromID(
    df, chrom_column="chrom", chrID_to_drop=["chrX", "chrY", "chrM"]
):
    """
    Filter a DataFrame based on chromosome IDs.

    This function takes a pandas DataFrame and a list of chromosome IDs
    to drop from the DataFrame. It filters out rows where the 'chrom' column
    matches any of the provided chromosome IDs.

    Parameters:
    - df (pandas.DataFrame): Input DataFrame containing a 'chrom' column
                            representing chromosome IDs.
    - chrID_to_drop (list): List of chromosome IDs to be filtered out from the DataFrame.

    Returns:
    pandas.DataFrame: A new DataFrame with rows removed where the 'chrom' column
                      matches any of the chromosome IDs in chrID_to_drop.
    """
    filtered_df = df[~df[chrom_column].isin(chrID_to_drop)]
    return filtered_df


def filter_by_chrmlen(df, chrom_sizes_file, buffer_bp=0):
    """
    Filters a DataFrame of dot intervals so that neither anchor exceeds
    chromosome size bounds (with optional buffer), using plain pandas.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: 'chrom', 'BIN1_START', 'BIN1_END', 'BIN2_START', 'BIN2_END'
    chrom_sizes_file : str or pd.DataFrame
        Path to chromosome sizes file (2-column TSV: chrom, size) or DataFrame
    buffer_bp : int
        Buffer to exclude near chromosome ends (in bp)

    Returns
    -------
    df_filtered : pd.DataFrame
        DataFrame with invalid rows removed
    """
    import pandas as pd

    assert isinstance(buffer_bp, int)

    # Read chrom sizes
    if isinstance(chrom_sizes_file, str):
        chrom_sizes_df = pd.read_csv(chrom_sizes_file, sep="\t", header=None, names=["chrom", "size"])
    else:
        chrom_sizes_df = chrom_sizes_file.rename(columns={0: "chrom", 1: "size"})

    # Convert to dictionary: {chrom: size}
    chrom_size_dict = dict(zip(chrom_sizes_df["chrom"], chrom_sizes_df["size"]))

    # Filtering function
    def is_valid(row):
        chrom = row['chrom']
        if chrom not in chrom_size_dict:
            return False
        chrom_len = chrom_size_dict[chrom]
        return (
            buffer_bp <= row['BIN1_START'] < row['BIN1_END'] <= chrom_len - buffer_bp and
            buffer_bp <= row['BIN2_START'] < row['BIN2_END'] <= chrom_len - buffer_bp
        )

    # Apply filtering
    df_filtered = df[df.apply(is_valid, axis=1)].copy()
    return df_filtered

In [5]:
# load dots (detected from HiC, 10kb resolution)
dots = pd.read_csv(dot_file, sep="\t")

In [6]:
len(dots)

9674

In [7]:
# dots[dots['BIN1_CHR'] != dots['BIN2_CHROMOSOME']]
# (sanity check) no interchromosomal dots
dots = dots.drop(columns=['BIN2_CHROMOSOME'])

# Rename BIN1_CHR to chrom
dots = dots.rename(columns={'BIN1_CHR': 'chrom'})

In [8]:
if autosomes_only:
    dots = filter_by_chromID(dots, chrID_to_drop=chromID_to_drop)

In [9]:
dots = filter_by_chrmlen(
    dots,
    chrom_sizes_file,
    seq_length,
)

In [10]:
def filter_by_anchor_distance(df, max_dist=384*2048):
    """
    Filters a DataFrame of dots to remove entries where the distance between
    BIN1_START and BIN2_START exceeds the maximum prediction window.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: 'BIN1_START', 'BIN2_START'
    max_dist : int
        Maximum allowed genomic distance between anchors (in bp)

    Returns
    -------
    df_filtered : pd.DataFrame
        Filtered DataFrame where abs(BIN1_START - BIN2_START) <= max_dist
    """
    df_filtered = df[abs(df['BIN1_START'] - df['BIN2_START']) <= max_dist].copy()
    return df_filtered

In [11]:
len(dots)

9251

In [12]:
dots_distance_filtered = filter_by_anchor_distance(dots, max_dist=384 * 2048) # 3/4th of the map

In [ ]:
384 * 2048

In [13]:
len(dots_distance_filtered)

6818

In [14]:
len(dots_distance_filtered) / len(dots)

0.7370014052534861

In [15]:
def enforce_anchor_order(df):
    """
    Ensures that BIN1_START < BIN2_START for each row.
    If not, swaps BIN1 and BIN2 columns.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: 'BIN1_START', 'BIN1_END', 'BIN2_START', 'BIN2_END'

    Returns
    -------
    df_ordered : pd.DataFrame
        DataFrame with anchors ordered so that BIN1_START <= BIN2_START
    """
    df = df.copy()
    swap_mask = df['BIN1_START'] > df['BIN2_START']

    # Swap BIN1 and BIN2 values where needed
    for col1, col2 in [('BIN1_START', 'BIN2_START'), ('BIN1_END', 'BIN2_END')]:
        df.loc[swap_mask, [col1, col2]] = df.loc[swap_mask, [col2, col1]].values

    return df

In [16]:
dots_distance_filtered = enforce_anchor_order(dots_distance_filtered)

In [17]:
dots_distance_filtered.reset_index(drop=True, inplace=True)

In [18]:
dots_distance_filtered

,chrom,BIN1_START,BIN1_END,BIN2_START,BIN2_END,FDR,DETECTION_SCALE
0,chr1,91970000,91980000,92450000,92460000,0.011590,2.111213
1,chr1,91970000,91980000,92610000,92620000,0.036990,2.599208
2,chr1,92160000,92170000,92450000,92460000,0.038089,2.111213
3,chr1,92490000,92500000,92610000,92620000,0.017336,3.200000
4,chr1,92680000,92690000,92850000,92860000,0.009500,4.222425
...,...,...,...,...,...,...,...
6813,chr19,36140000,36150000,36440000,36450000,0.013890,6.400000
6814,chr19,36180000,36190000,36430000,36440000,0.000199,2.111213
6815,chr19,37050000,37060000,37260000,37270000,0.030770,2.111213
6816,chr19,37550000,37560000,37930000,37940000,0.037086,2.599208


## Overlapping with JASPAR CTCFs

In [19]:
jaspar_file = "/project/fudenber_735/motifs/mm10/jaspar/MA0139.1.tsv.gz"

In [20]:
# read jaspar table (all CTCFs in the mouse genome)
jaspar_df = bf.read_table(jaspar_file, schema="jaspar", skiprows=1)
if autosomes_only:
    jaspar_df = filter_by_chromID(jaspar_df, chrID_to_drop=chromID_to_drop)
jaspar_df.reset_index(drop=True, inplace=True)

In [21]:
import pybedtools

In [22]:
dots_df = dots_distance_filtered.rename(columns={
    'BIN1_START': 'anchor1_start',
    'BIN1_END': 'anchor1_end',
    'BIN2_START': 'anchor2_start',
    'BIN2_END': 'anchor2_end'
})

In [23]:
# Anchor 1 BED
anchor1_bed = dots_df[['chrom', 'anchor1_start', 'anchor1_end']].copy()
anchor1_bed['name'] = dots_df.index  # Use index to track original row
anchor1_bed.columns = ['chrom', 'start', 'end', 'name']

# Anchor 2 BED
anchor2_bed = dots_df[['chrom', 'anchor2_start', 'anchor2_end']].copy()
anchor2_bed['name'] = dots_df.index
anchor2_bed.columns = ['chrom', 'start', 'end', 'name']

# CTCF BED
ctcf_bed = jaspar_df[['chrom', 'start', 'end', 'strand']].copy()
ctcf_bed['name'] = ctcf_bed.index
ctcf_bed.columns = ['chrom', 'start', 'end', 'strand', 'name']

In [24]:
a1_bt = pybedtools.BedTool.from_dataframe(anchor1_bed)
a2_bt = pybedtools.BedTool.from_dataframe(anchor2_bed)
ctcf_bt = pybedtools.BedTool.from_dataframe(ctcf_bed)

In [25]:
# intersect anchors with ctcfs
# Anchor1 intersects
a1_intersect = a1_bt.intersect(ctcf_bt, wa=True, wb=True)
a1_df = a1_intersect.to_dataframe(names=[
    'a_chrom', 'a_start', 'a_end', 'row_id',
    'ctcf_chrom', 'ctcf_start', 'ctcf_end', 'ctcf_strand', 'ctcf_id'
])
a1_df = a1_df.sort_values(['row_id', 'ctcf_start'])
a1_df = a1_df[['row_id', 'ctcf_strand']].groupby('row_id').agg(lambda x: ','.join(set(x))).rename(columns={'ctcf_strand': 'anchor1_ctcf_strand'})

# Anchor2 intersects
a2_intersect = a2_bt.intersect(ctcf_bt, wa=True, wb=True)
a2_df = a2_intersect.to_dataframe(names=[
    'a_chrom', 'a_start', 'a_end', 'row_id',
    'ctcf_chrom', 'ctcf_start', 'ctcf_end', 'ctcf_strand', 'ctcf_id'
])
a2_df = a2_df.sort_values(['row_id', 'ctcf_start'])
a2_df = a2_df[['row_id', 'ctcf_strand']].groupby('row_id').agg(lambda x: ','.join(set(x))).rename(columns={'ctcf_strand': 'anchor2_ctcf_strand'})

In [26]:
dots_df['row_id'] = dots_df.index.astype(int)
a1_df.index = a1_df.index.astype(int)
a2_df.index = a2_df.index.astype(int)

# If a1_df and a2_df use index as row_id, you can reset:
a1_df = a1_df.reset_index()
a2_df = a2_df.reset_index()

In [27]:
dots_annotated = dots_df.merge(a1_df, on='row_id', how='left').merge(a2_df, on='row_id', how='left')

# Fill NaNs
dots_annotated['anchor1_ctcf_strand'] = dots_annotated['anchor1_ctcf_strand'].fillna('None')
dots_annotated['anchor2_ctcf_strand'] = dots_annotated['anchor2_ctcf_strand'].fillna('None')


In [28]:
dots_annotated

,chrom,anchor1_start,anchor1_end,anchor2_start,anchor2_end,FDR,DETECTION_SCALE,row_id,anchor1_ctcf_strand,anchor2_ctcf_strand
0,chr1,91970000,91980000,92450000,92460000,0.011590,2.111213,0,"-,+","-,+"
1,chr1,91970000,91980000,92610000,92620000,0.036990,2.599208,1,"-,+","-,+"
2,chr1,92160000,92170000,92450000,92460000,0.038089,2.111213,2,"-,+","-,+"
3,chr1,92490000,92500000,92610000,92620000,0.017336,3.200000,3,"-,+","-,+"
4,chr1,92680000,92690000,92850000,92860000,0.009500,4.222425,4,"-,+","-,+"
...,...,...,...,...,...,...,...,...,...,...
6813,chr19,36140000,36150000,36440000,36450000,0.013890,6.400000,6813,"-,+","-,+"
6814,chr19,36180000,36190000,36430000,36440000,0.000199,2.111213,6814,"-,+","-,+"
6815,chr19,37050000,37060000,37260000,37270000,0.030770,2.111213,6815,"-,+","-,+"
6816,chr19,37550000,37560000,37930000,37940000,0.037086,2.599208,6816,"-,+",-


In [29]:
# 2nd dot anchor is not moving -> always centered at 384th bin
# since dots were called at 10kb resolution - let's assume 10kb -> ~5bins
# then the 2nd anchor starts with 382nd bin 

bin_size = 2048
cropping_applied = 64

rel_2anchor_start = (382 + cropping_applied) * bin_size
rel_2anchor_end = rel_2anchor_start + bin_size * 5

In [30]:
to_window_end = seq_length - rel_2anchor_start

In [31]:
dots_annotated.columns

Index(['chrom', 'anchor1_start', 'anchor1_end', 'anchor2_start', 'anchor2_end',
       'FDR', 'DETECTION_SCALE', 'row_id', 'anchor1_ctcf_strand',
       'anchor2_ctcf_strand'],
      dtype='object')

In [32]:
dots_annotated["window_start"] = dots_annotated["anchor2_start"] - rel_2anchor_start
dots_annotated["window_end"] = dots_annotated["anchor2_start"] + to_window_end

In [33]:
(dots_annotated["window_end"] - dots_annotated["window_start"] != 1310720).any()

False

In [34]:
dots_annotated["anchor2_center_bin"] = [384 for i in range(len(dots_annotated))]

# now, based on the dot1 - dot2 distance, we'll figure out which bin is center of the other dot

In [35]:
dots_annotated["anchors_dist"] = (dots_annotated["anchor2_start"] - dots_annotated["anchor1_start"]) // bin_size

In [36]:
dots_annotated["anchor1_center_bin"] = dots_annotated["anchor2_center_bin"] - dots_annotated["anchors_dist"]

In [37]:
# sanity check
dots_annotated[dots_annotated["anchor1_center_bin"] < 0]

,chrom,anchor1_start,anchor1_end,anchor2_start,anchor2_end,FDR,DETECTION_SCALE,row_id,anchor1_ctcf_strand,anchor2_ctcf_strand,window_start,window_end,anchor2_center_bin,anchors_dist,anchor1_center_bin


In [38]:
dots_annotated

,chrom,anchor1_start,anchor1_end,anchor2_start,anchor2_end,FDR,DETECTION_SCALE,row_id,anchor1_ctcf_strand,anchor2_ctcf_strand,window_start,window_end,anchor2_center_bin,anchors_dist,anchor1_center_bin
0,chr1,91970000,91980000,92450000,92460000,0.011590,2.111213,0,"-,+","-,+",91536592,92847312,384,234,150
1,chr1,91970000,91980000,92610000,92620000,0.036990,2.599208,1,"-,+","-,+",91696592,93007312,384,312,72
2,chr1,92160000,92170000,92450000,92460000,0.038089,2.111213,2,"-,+","-,+",91536592,92847312,384,141,243
3,chr1,92490000,92500000,92610000,92620000,0.017336,3.200000,3,"-,+","-,+",91696592,93007312,384,58,326
4,chr1,92680000,92690000,92850000,92860000,0.009500,4.222425,4,"-,+","-,+",91936592,93247312,384,83,301
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6813,chr19,36140000,36150000,36440000,36450000,0.013890,6.400000,6813,"-,+","-,+",35526592,36837312,384,146,238
6814,chr19,36180000,36190000,36430000,36440000,0.000199,2.111213,6814,"-,+","-,+",35516592,36827312,384,122,262
6815,chr19,37050000,37060000,37260000,37270000,0.030770,2.111213,6815,"-,+","-,+",36346592,37657312,384,102,282
6816,chr19,37550000,37560000,37930000,37940000,0.037086,2.599208,6816,"-,+",-,37016592,38327312,384,185,199


In [39]:
dots_annotated.to_csv('/scratch1/smaruj/natural_dots/filtered_dots_with_orientations.tsv', sep='\t', index=False)